# 3-2 折れ線と散布図

3-1 の **棒グラフ** は「項目別の大きさを比べる」のに向いていました。今回扱うのは:

- **折れ線** — **時間の経過に沿った推移** を見るのに最適（売上の月次推移、気温の変化など）
- **散布図** — **2 つの数値の関係** を見るのに最適（数量 × 売上、広告費 × 来店数 など）

## このノートのゴール

- `plt.plot(x, y)` で **折れ線** を描ける
- `marker` / `linestyle` / `color` で線の見た目を変えられる
- **複数の系列** を 1 枚に並べて、**`plt.legend()` で凡例** を付けられる
- `plt.scatter(x, y)` で **散布図** を描き、2 つの数値の関係を見られる
- 棒・折れ線・散布図の **使い分け** が分かる

## Excel との対応表

| やりたいこと | Excel | matplotlib |
|---|---|---|
| 折れ線 | グラフの挿入 → 折れ線 | `plt.plot(x, y)` |
| マーカー付き折れ線 | グラフ要素 → データマーカー | `plt.plot(x, y, marker="o")` |
| 点線 | 書式設定 → 線の種類 | `plt.plot(..., linestyle="--")` |
| 散布図 | グラフの挿入 → 散布図 | `plt.scatter(x, y)` |
| 凡例 | グラフ要素 → 凡例 | `plt.legend()` |
| グリッド | グラフ要素 → 目盛線 | `plt.grid(True)` |

## 準備

In [ ]:
!pip install -q japanize-matplotlib

import matplotlib.pyplot as plt
import japanize_matplotlib
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/python-data-basics"
DATA = f"{BASE}/data/pandas"

df = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上", parse_dates=["date"])
df.head()

## 1. 折れ線グラフの基本 — `plt.plot`

棒グラフが `plt.bar` だったのと同じ感覚で、折れ線は **`plt.plot(x, y)`** です。

日次の売上合計を折れ線にして、月の中で「いつが山だったか」を見てみます。

In [ ]:
# 日付ごとの売上合計（→ index が日付の Series）
daily = df.groupby("date")["amount"].sum()
daily.head()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(daily.index, daily.values)
plt.title("2026年1月 日次売上推移")
plt.xlabel("日付")
plt.ylabel("売上 (円)")
plt.xticks(rotation=45)
plt.show()

## 2. マーカー・線種・色

`plt.plot` には見た目を変えるオプションがあります。

| オプション | 効果 | よく使う値 |
|---|---|---|
| `marker` | 各点に印を付ける | `"o"` 丸、`"s"` 四角、`"^"` 三角、`"x"` バツ |
| `linestyle` | 線の種類 | `"-"` 実線（既定）、`"--"` 破線、`":"` 点線 |
| `color` | 線とマーカーの色 | `"red"` `"steelblue"` など |
| `linewidth` | 線の太さ | 数値（既定 1.5 くらい）|

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(daily.index, daily.values,
         marker="o", linestyle="--", color="steelblue", linewidth=1.5)
plt.title("2026年1月 日次売上推移")
plt.xlabel("日付")
plt.ylabel("売上 (円)")
plt.xticks(rotation=45)
plt.show()

## 3. 複数の系列を 1 枚に並べる — `plt.legend`

「**商品ごとに** 日次推移を比べたい」というときは、`plt.plot` を **何度か呼ぶ** だけで、自動的に同じ図に重ね描きされます。

それぞれに **`label="..."`** を付けて、最後に **`plt.legend()`** を呼ぶと **凡例** が出ます。

ここでは 1-4 で習った **`for` ループ** と 2-3 の **boolean indexing + groupby** を組み合わせます。

In [ ]:
plt.figure(figsize=(10, 5))

# 商品ごとに線を1本ずつ追加
for name in df["product_name"].unique():
    sub = df[df["product_name"] == name].groupby("date")["amount"].sum()
    plt.plot(sub.index, sub.values, marker="o", label=name)

plt.title("商品別 日次売上推移")
plt.xlabel("日付")
plt.ylabel("売上 (円)")
plt.xticks(rotation=45)
plt.legend()   # ← 各 plot の label を集めて凡例にしてくれる
plt.show()

**ポイント**:

- `df["product_name"].unique()` で「重複を除いた商品名のリスト」を取得（2-3 の補足）
- 線が重なってごちゃつくときは、`figsize` を大きくする・特定の商品だけに絞るなどで調整
- **`plt.legend()` を呼ばないと凡例は出ない** ので忘れずに

## 4. グリッド (目盛線) を付ける

折れ線グラフは **目盛線** があると数値が読み取りやすくなります。1行追加するだけです。

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(daily.index, daily.values, marker="o")
plt.title("2026年1月 日次売上推移")
plt.xlabel("日付")
plt.ylabel("売上 (円)")
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.5)   # ← 薄い破線のグリッド
plt.show()

## 5. 散布図 — `plt.scatter`

**2 つの数値の関係** を見たいときは散布図です。Excel の「グラフの挿入 → 散布図」に対応します。

ここでは **「数量」と「売上金額」の関係** を見てみます。

書き方は `plt.scatter(x, y)`。

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["quantity"], df["amount"])
plt.title("数量 × 売上金額")
plt.xlabel("数量")
plt.ylabel("売上金額 (円)")
plt.grid(True, alpha=0.3)
plt.show()

数量が増えるほど売上も増える **正の関係** が見えます。点が **何本かの直線** に並んでいるのは、商品ごとに単価が違うため（`amount = quantity × unit_price` なので、同じ単価の商品は同じ傾きの直線に乗る）。

### 商品別に色分けする

In [ ]:
plt.figure(figsize=(8, 5))

# 商品ごとに色分け
for name in df["product_name"].unique():
    sub = df[df["product_name"] == name]
    plt.scatter(sub["quantity"], sub["amount"], label=name)

plt.title("商品別 数量 × 売上金額")
plt.xlabel("数量")
plt.ylabel("売上金額 (円)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. 棒・折れ線・散布図の使い分け

| 何を見たい？ | 例 | 適切なグラフ |
|---|---|---|
| **項目別の大きさを比べたい** | 商品別の売上、部署別の人数 | 棒グラフ |
| **時間の経過に沿った推移** | 月次売上、日次気温、株価 | 折れ線 |
| **2 つの数値の関係を見たい** | 数量×売上、広告費×来店数、身長×体重 | 散布図 |
| **割合を見たい** | 部署別売上構成比 | 円グラフ（3-4 で軽く扱う）|

> 💡 迷ったら **「項目（カテゴリ）か、時間か、数値同士か」** で考えると選びやすいです。

## 練習問題

上で読み込んだ `df` を使って、以下を描いてください。

1. **日次売上推移** を折れ線で描いてください。マーカーは `"^"` (三角)、色は `"darkgreen"`、線は破線 `"--"`、グリッドあり。
2. **「メロン」と「いちご (1パック)」の 2 商品だけ** を、1 枚の折れ線グラフに並べてください（凡例付き）。
3. **単価 (`unit_price`) × 数量 (`quantity`)** の散布図を描いてください。「単価が高い商品ほど、1回あたりの購入数量が少なくなる傾向」が見えるかチェック。

In [ ]:
# ここに書いてみてください



<details>
<summary>答えを見る</summary>

```python
# 1. 日次売上推移（カスタム見た目）
daily = df.groupby("date")["amount"].sum()

plt.figure(figsize=(10, 4))
plt.plot(daily.index, daily.values,
         marker="^", linestyle="--", color="darkgreen")
plt.title("日次売上推移")
plt.xlabel("日付")
plt.ylabel("売上 (円)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.4)
plt.show()

# 2. メロンといちごの折れ線比較
targets = ["メロン", "いちご (1パック)"]

plt.figure(figsize=(10, 4))
for name in targets:
    sub = df[df["product_name"] == name].groupby("date")["amount"].sum()
    plt.plot(sub.index, sub.values, marker="o", label=name)

plt.title("メロン vs いちご 日次売上")
plt.xlabel("日付")
plt.ylabel("売上 (円)")
plt.xticks(rotation=45)
plt.legend()
plt.show()

# 3. 単価 × 数量の散布図
plt.figure(figsize=(7, 5))
plt.scatter(df["unit_price"], df["quantity"], alpha=0.5)
plt.title("単価 × 1回あたり数量")
plt.xlabel("単価 (円)")
plt.ylabel("数量")
plt.grid(True, alpha=0.3)
plt.show()
# → 単価が高い (メロン 2800円など) 商品は数量が少なく、
#    安い (ばなな 280円など) 商品は数量がばらつく、という傾向が見える
```

</details>

## よくあるエラー

### 1. 凡例が出ない
→ `plt.plot(..., label="...")` で **label を付け忘れ**、または **`plt.legend()` 呼び忘れ**。両方必要。

### 2. 線が重ならず別々に出てしまう
→ 各 `plt.plot` の間で `plt.show()` を呼んでいる。`plt.show()` は **最後に 1 回だけ**。

### 3. 折れ線が左右に飛び跳ねる（順番が狂う）
→ `groupby("date")` で **インデックスがソートされた Series** が返るので普通は問題ないが、`df.sort_values("date")` を挟むのも安全。

### 4. 散布図の点が重なって見えない
→ `plt.scatter(..., alpha=0.5)` で **半透明** にすると重なりが分かる。

### 5. `plt.plot(x, y)` で `ValueError: x and y must have same first dimension`
→ x と y の **長さ（要素数）が違う**。`len(x)` と `len(y)` を `print` で確認。

## まとめ

- **折れ線** = `plt.plot(x, y)`、時間の経過に沿った推移に最適
- `marker` / `linestyle` / `color` / `linewidth` で見た目をカスタマイズ
- **複数系列** は `plt.plot` を for で何度も呼んで `label` を付け、最後に **`plt.legend()`**
- `plt.grid(True)` で目盛線、`alpha` で透明度
- **散布図** = `plt.scatter(x, y)`、2 つの数値の関係を見るのに最適
- グラフは目的に応じて **棒 / 折れ線 / 散布図** を選ぶ

次は **3-3「pandas から直接グラフを描く」** で、ここまで `plt.bar(...)` `plt.plot(...)` と書いてきたものを、**`df.plot(...)`** で更に短く書く方法を扱います。